# AgroCheck — Separar as fotos de TESTE (que a IA nunca viu)

Este caderno refaz **exatamente a mesma separação** do treino (mesma ordem, mesma semente `random.seed(42)`) e copia para o seu Drive só as fotos que ficaram no **conjunto de teste**, com o **nome original** de cada arquivo.

Resultado em **Meu Drive → AgroCheck_fotos_teste**:
- uma pasta para cada classe, com as fotos de teste;
- o arquivo `lista_fotos_teste.csv` com o nome de todas elas.

**Não precisa de GPU** e não treina nada. É só rodar a célula abaixo (▶). O `dataset.zip` precisa estar em Meu Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import zipfile, os, re, random, csv

CAMINHO_ZIP = "/content/drive/MyDrive/dataset.zip"
SAIDA = "/content/drive/MyDrive/AgroCheck_fotos_teste"

# IGUAL ao caderno de treino (mesma ordem e mesmos nomes)
CLASSES = [
    "Corn___Common_Rust_Leaf",
    "Corn___Gray_Leaf_Spot_Leaf",
    "Corn___Healthy_Leaf",
    "Corn___Northern_Leaf_Blight_Leaf",
    "Potato___Early_blight_Leaf",
    "Potato___Late_Blight_Leaf",
    "Potato__Healthy_Leaf",
]
NOMES_PT = [
    "Milho - Ferrugem comum",
    "Milho - Mancha cinzenta",
    "Milho saudável",
    "Milho - Helmintosporiose",
    "Batata - Pinta-preta",
    "Batata - Requeima",
    "Batata saudável",
]
EXTENSOES = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def normalizar(nome):
    return re.sub(r"_+", "_", nome).lower()

def eh_da_classe(caminho, classe):
    partes = caminho.replace("\\", "/").split("/")[:-1]
    return normalizar(classe) in [normalizar(p) for p in partes]

z = zipfile.ZipFile(CAMINHO_ZIP)
todos = [n for n in z.namelist() if n.lower().endswith(EXTENSOES)]

os.makedirs(SAIDA, exist_ok=True)
linhas = []

random.seed(42)   # mesma semente do treino
for classe, nome_pt in zip(CLASSES, NOMES_PT):
    arquivos = sorted(n for n in todos if eh_da_classe(n, classe))
    random.shuffle(arquivos)          # mesmo embaralhamento do treino
    n = len(arquivos)
    n_treino = int(n * 0.70)
    n_val = int(n * 0.15)
    teste = arquivos[n_treino + n_val:]   # os últimos 15% = conjunto de teste

    pasta = os.path.join(SAIDA, nome_pt)
    os.makedirs(pasta, exist_ok=True)
    for caminho in teste:
        nome_arquivo = os.path.basename(caminho)
        with open(os.path.join(pasta, nome_arquivo), "wb") as f:
            f.write(z.read(caminho))
        linhas.append([nome_pt, classe, nome_arquivo, caminho])
    print(f"{nome_pt}: {len(teste)} fotos de teste copiadas")

with open(os.path.join(SAIDA, "lista_fotos_teste.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["classe_no_site", "pasta_no_dataset", "arquivo", "caminho_no_zip"])
    w.writerows(linhas)

print(f"\nPronto! {len(linhas)} fotos em Meu Drive → AgroCheck_fotos_teste ✅")